# Phase 3 — Exploratory Data Analysis

This notebook performs deep exploratory analysis of the Customer Support on Twitter dataset.

**Goal:** Understand the data, identify quality problems, and prepare for brand selection.

**Important:** This notebook auto-detects the dataset schema. No column names are assumed.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.eda_utils import (
    calculate_brand_statistics,
    calculate_conversation_statistics,
    calculate_duplicate_statistics,
    calculate_response_time_statistics,
    calculate_text_statistics,
    detect_columns,
    detect_noise_features,
    summarize_dataframe,
    summarize_missing_values,
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
eda_stats: dict = {}

## 1. Load Dataset & Detect Schema

In [ ]:
# Find dataset
dev_sample = INTERIM_DIR / 'development_sample.csv'
raw_csvs = sorted(RAW_DIR.glob('*.csv'))

if dev_sample.exists():
    data_path = dev_sample
    print(f'Using development sample: {dev_sample}')
elif raw_csvs:
    data_path = raw_csvs[0]
    print(f'Using raw dataset: {raw_csvs[0].name}')
else:
    data_path = None
    print('No dataset found. Run Phase 2 scripts first.')

if data_path:
    # Load in chunks for large files
    chunks = []
    for chunk in pd.read_csv(data_path, chunksize=50000, low_memory=False):
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True)
    print(f'Loaded {len(df):,} rows, {len(df.columns)} columns.')
    
    # Detect schema
    col_map = detect_columns(df)
    print(f'\nDetected column mapping:')
    for logical, actual in col_map.items():
        if actual:
            print(f'  {logical}: {actual}')

In [ ]:
if 'df' in dir():
    # Basic info
    summary = summarize_dataframe(df)
    eda_stats['dataset'] = summary
    print(f'Rows: {summary["rows"]:,}')
    print(f'Columns: {summary["columns"]}')
    print(f'Memory: {summary["memory_mb"]} MB')
    print(f'\nColumn names: {summary["column_names"]}')
    print(f'\nData types:')
    for col, dtype in summary['dtypes'].items():
        print(f'  {col}: {dtype}')
    
    display(df.head(3))

## 2. Dataset-Level Overview

In [ ]:
if 'df' in dir():
    overview = {'messages': len(df)}
    
    conv_col = col_map.get('conversation_id')
    if conv_col and conv_col in df.columns:
        conv_stats = calculate_conversation_statistics(df, conv_col)
        overview['conversations'] = conv_stats['unique_conversations']
        overview['median_conversation_length'] = conv_stats['median_length']
        overview['max_conversation_length'] = conv_stats['max_length']
        overview['single_message_pct'] = conv_stats['single_message_pct']
        overview['multi_message_pct'] = conv_stats['multi_message_pct']
        eda_stats['conversations'] = conv_stats
        
        print(f'Conversations: {overview["conversations"]:,}')
        print(f'Median conversation length: {overview["median_conversation_length"]}')
        print(f'Max conversation length: {overview["max_conversation_length"]}')
        print(f'Single-message conversations: {overview["single_message_pct"]}%')
        print(f'Multi-message conversations: {overview["multi_message_pct"]}%')
    else:
        print('No conversation ID column detected.')
    
    brand_col = col_map.get('brand')
    if brand_col and brand_col in df.columns:
        overview['brands'] = int(df[brand_col].nunique())
        print(f'Brands: {overview["brands"]:,}')
    
    author_col = col_map.get('author_id')
    if author_col and author_col in df.columns:
        overview['unique_users'] = int(df[author_col].nunique())
        print(f'Unique users: {overview["unique_users"]:,}')
    
    time_col = col_map.get('created_at')
    if time_col and time_col in df.columns:
        try:
            times = pd.to_datetime(df[time_col], errors='coerce')
            valid_times = times.dropna()
            if len(valid_times) > 0:
                overview['date_range'] = f'{valid_times.min()} to {valid_times.max()}'
                print(f'Date range: {overview["date_range"]}')
        except Exception:
            pass
    
    eda_stats['overview'] = overview
    
    # Summary table
    print('\n--- Dataset Summary ---')
    for k, v in overview.items():
        print(f'{k}: {v}')

## 3. Missing Values

In [ ]:
if 'df' in dir():
    missing = summarize_missing_values(df)
    eda_stats['missingness'] = missing
    
    # Show columns with missing values
    missing_cols = {k: v for k, v in missing.items() if v['missing_count'] > 0}
    
    if missing_cols:
        print('Columns with missing values:')
        for col, info in sorted(missing_cols.items(), key=lambda x: -x[1]['missing_pct']):
            print(f'  {col}: {info["missing_count"]:,} ({info["missing_pct"]}%)')
        
        # Plot
        miss_df = pd.DataFrame(missing_cols).T.sort_values('missing_pct', ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(4, len(miss_df) * 0.4)))
        miss_df['missing_pct'].plot(kind='barh', ax=ax, color='coral')
        ax.set_xlabel('Missing %')
        ax.set_title('Missing Values by Column')
        plt.tight_layout()
        plt.show()
    else:
        print('No missing values found.')

## 4. Brand Distribution Analysis

In [ ]:
if 'df' in dir():
    brand_col = col_map.get('brand')
    conv_col = col_map.get('conversation_id')
    inbound_col = col_map.get('inbound')
    
    if brand_col and brand_col in df.columns:
        brand_stats = calculate_brand_statistics(df, brand_col, conv_col)
        eda_stats['brands'] = brand_stats
        
        print(f'Unique brands: {brand_stats["unique_brands"]}')
        
        # Top brands by message count
        top_brands = pd.Series(brand_stats['top_brands']).head(20)
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        
        # Chart 1: Top brands by messages
        top_brands.plot(kind='barh', ax=axes[0], color='steelblue')
        axes[0].set_xlabel('Number of Messages')
        axes[0].set_title('Top 20 Brands by Message Count')
        axes[0].invert_yaxis()
        
        # Chart 2: Top brands by conversations
        if 'conversations_per_brand' in brand_stats:
            conv_per_brand = pd.Series(brand_stats['conversations_per_brand']).head(20)
            conv_per_brand.plot(kind='barh', ax=axes[1], color='darkorange')
            axes[1].set_xlabel('Number of Conversations')
            axes[1].set_title('Top 20 Brands by Conversation Count')
            axes[1].invert_yaxis()
        
        plt.tight_layout()
        plt.show()
        
        # Chart 3: Conversation length by top brands
        if conv_col and conv_col in df.columns:
            top_10_brands = top_brands.head(10).index.tolist()
            df_top = df[df[brand_col].isin(top_10_brands)].copy()
            conv_lengths = df_top.groupby([brand_col, conv_col]).size().reset_index(name='length')
            
            fig, ax = plt.subplots(figsize=(14, 6))
            order = conv_lengths.groupby(brand_col)['length'].median().sort_values(ascending=False).index
            sns.boxplot(data=conv_lengths, x=brand_col, y='length', order=order, ax=ax, showfliers=False)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
            ax.set_ylabel('Conversation Length')
            ax.set_title('Conversation Length Distribution by Top 10 Brands')
            plt.tight_layout()
            plt.show()
    else:
        print('No brand column detected.')

## 5. Candidate Brand Identification

In [ ]:
if 'df' in dir():
    brand_col = col_map.get('brand')
    conv_col = col_map.get('conversation_id')
    inbound_col = col_map.get('inbound')
    
    if brand_col and brand_col in df.columns and conv_col and conv_col in df.columns:
        # Build candidate table
        candidates = []
        
        for brand, group in df.groupby(brand_col):
            n_convs = group[conv_col].nunique()
            n_msgs = len(group)
            
            # Customer vs support messages
            if inbound_col and inbound_col in group.columns:
                inbound_mask = group[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
                n_customer = int(inbound_mask.sum())
                n_support = int((~inbound_mask).sum())
            else:
                n_customer = None
                n_support = None
            
            # Multi-turn conversations
            conv_sizes = group.groupby(conv_col).size()
            n_multi = int((conv_sizes > 1).sum())
            multi_pct = round(n_multi / n_convs * 100, 1) if n_convs > 0 else 0
            
            # Response coverage
            if inbound_col and inbound_col in group.columns:
                convs_with_support = group[~inbound_mask].groupby(conv_col).ngroups
                response_coverage = round(convs_with_support / n_convs * 100, 1) if n_convs > 0 else 0
            else:
                response_coverage = None
            
            candidates.append({
                'brand': brand,
                'conversations': n_convs,
                'messages': n_msgs,
                'customer_messages': n_customer,
                'support_messages': n_support,
                'multi_turn_pct': multi_pct,
                'avg_conv_length': round(n_msgs / n_convs, 1) if n_convs > 0 else 0,
                'response_coverage': response_coverage,
            })
        
        candidate_df = pd.DataFrame(candidates).sort_values('conversations', ascending=False)
        eda_stats['brand_candidates'] = candidate_df.head(20).to_dict(orient='records')
        
        print('Top 20 Candidate Brands:')
        display(candidate_df.head(20).style.format({
            'conversations': '{:,}',
            'messages': '{:,}',
            'customer_messages': '{:,}',
            'support_messages': '{:,}',
        }))
    else:
        print('Brand or conversation column not detected.')

## 6. Conversation-Length Analysis

In [ ]:
if 'df' in dir():
    conv_col = col_map.get('conversation_id')
    
    if conv_col and conv_col in df.columns:
        conv_lengths = df.groupby(conv_col).size()
        
        # Distribution
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        conv_lengths.clip(upper=50).hist(bins=50, ax=axes[0], edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Messages per Conversation')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Conversation Length Distribution')
        axes[0].axvline(conv_lengths.median(), color='red', linestyle='--', 
                       label=f'Median: {conv_lengths.median():.0f}')
        axes[0].legend()
        
        # Bucketed
        buckets = {
            '1 msg': int((conv_lengths == 1).sum()),
            '2 msgs': int((conv_lengths == 2).sum()),
            '3 msgs': int((conv_lengths == 3).sum()),
            '4 msgs': int((conv_lengths == 4).sum()),
            '5+ msgs': int((conv_lengths >= 5).sum()),
        }
        pd.Series(buckets).plot(kind='bar', ax=axes[1], color='teal', edgecolor='black')
        axes[1].set_ylabel('Number of Conversations')
        axes[1].set_title('Conversation Length Buckets')
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
        
        plt.tight_layout()
        plt.show()
        
        # Percentiles
        conv_stats = {
            'mean': round(float(conv_lengths.mean()), 1),
            'median': round(float(conv_lengths.median()), 1),
            'p75': round(float(conv_lengths.quantile(0.75)), 1),
            'p90': round(float(conv_lengths.quantile(0.90)), 1),
            'p95': round(float(conv_lengths.quantile(0.95)), 1),
            'max': int(conv_lengths.max()),
        }
        eda_stats['conversation_length_distribution'] = conv_stats
        
        print('Conversation length statistics:')
        for k, v in conv_stats.items():
            print(f'  {k}: {v}')
        
        # Investigate very long conversations
        long_convs = conv_lengths[conv_lengths > conv_lengths.quantile(0.99)]
        if len(long_convs) > 0:
            print(f'\nVery long conversations (> p99 = {conv_lengths.quantile(0.99):.0f} msgs):')
            print(f'  Count: {len(long_convs)}')
            print(f'  Max: {long_convs.max()}')
    else:
        print('No conversation ID column detected.')

## 7. Customer vs Support-Agent Behavior

In [ ]:
if 'df' in dir():
    inbound_col = col_map.get('inbound')
    text_col = col_map.get('text')
    
    if inbound_col and inbound_col in df.columns and text_col and text_col in df.columns:
        inbound_mask = df[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
        
        customer_msgs = df[inbound_mask][text_col]
        support_msgs = df[~inbound_mask][text_col]
        
        print(f'Customer messages: {len(customer_msgs):,}')
        print(f'Support messages: {len(support_msgs):,}')
        print(f'Customer/Support ratio: {len(customer_msgs)/len(support_msgs):.2f}')
        
        # Length comparison
        cust_len = customer_msgs.str.len()
        supp_len = support_msgs.str.len()
        
        behavior_stats = {
            'customer_count': int(len(customer_msgs)),
            'support_count': int(len(support_msgs)),
            'customer_mean_length': round(float(cust_len.mean()), 1),
            'support_mean_length': round(float(supp_len.mean()), 1),
            'customer_median_length': round(float(cust_len.median()), 1),
            'support_median_length': round(float(supp_len.median()), 1),
        }
        eda_stats['behavior'] = behavior_stats
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        cust_len.clip(upper=500).hist(bins=50, ax=axes[0], alpha=0.7, label='Customer', color='steelblue')
        supp_len.clip(upper=500).hist(bins=50, ax=axes[0], alpha=0.7, label='Support', color='darkorange')
        axes[0].set_xlabel('Message Length (chars)')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Message Length: Customer vs Support')
        axes[0].legend()
        
        # Support responses with links
        supp_with_urls = support_msgs.str.contains(r'https?://', regex=True, na=False).mean() * 100
        cust_with_urls = customer_msgs.str.contains(r'https?://', regex=True, na=False).mean() * 100
        
        url_stats = pd.Series({
            'Customer': cust_with_urls,
            'Support': supp_with_urls,
        })
        url_stats.plot(kind='bar', ax=axes[1], color=['steelblue', 'darkorange'], edgecolor='black')
        axes[1].set_ylabel('% of Messages with URLs')
        axes[1].set_title('URL Usage: Customer vs Support')
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
        
        plt.tight_layout()
        plt.show()
        
        print(f'\nCustomer mean length: {behavior_stats["customer_mean_length"]} chars')
        print(f'Support mean length: {behavior_stats["support_mean_length"]} chars')
        print(f'Support messages with URLs: {supp_with_urls:.1f}%')
    else:
        print('Inbound or text column not detected.')

## 8. Response-Time Analysis

In [ ]:
if 'df' in dir():
    conv_col = col_map.get('conversation_id')
    time_col = col_map.get('created_at')
    inbound_col = col_map.get('inbound')
    
    if all(c and c in df.columns for c in [conv_col, time_col, inbound_col]):
        rt_stats = calculate_response_time_statistics(df, conv_col, time_col, inbound_col)
        
        if rt_stats:
            eda_stats['response_times'] = rt_stats
            print(f'Response time analysis (based on {rt_stats["sample_size"]:,} conversations):')
            print(f'  Median: {rt_stats["median_seconds"]:.1f}s ({rt_stats["median_seconds"]/60:.1f} min)')
            print(f'  Mean: {rt_stats["mean_seconds"]:.1f}s ({rt_stats["mean_seconds"]/60:.1f} min)')
            print(f'  P75: {rt_stats["p75_seconds"]:.1f}s')
            print(f'  P90: {rt_stats["p90_seconds"]:.1f}s')
            print(f'  P95: {rt_stats["p95_seconds"]:.1f}s')
            
            # Visualize (limited to reasonable range)
            df_rt = df.copy()
            df_rt['_time'] = pd.to_datetime(df_rt[time_col], errors='coerce')
            valid = df_rt.dropna(subset=['_time'])
            
            inbound_mask = valid[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
            first_inbound = valid[inbound_mask].groupby(conv_col)['_time'].min()
            first_outbound = valid[~inbound_mask].groupby(conv_col)['_time'].min()
            
            common = set(first_inbound.index) & set(first_outbound.index)
            rts = []
            for cid in list(common)[:10000]:
                delta = (first_outbound[cid] - first_inbound[cid]).total_seconds()
                if 0 <= delta < 86400:  # Within 24 hours
                    rts.append(delta / 60)  # Convert to minutes
            
            if rts:
                fig, ax = plt.subplots(figsize=(10, 5))
                pd.Series(rts).clip(upper=120).hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
                ax.set_xlabel('Response Time (minutes)')
                ax.set_ylabel('Count')
                ax.set_title('First Support Response Time Distribution')
                ax.axvline(np.median(rts), color='red', linestyle='--', label=f'Median: {np.median(rts):.1f} min')
                ax.legend()
                plt.tight_layout()
                plt.show()
        else:
            print('Could not compute response times (insufficient data).')
    else:
        print('Missing required columns for response-time analysis.')
        print('Need: conversation_id, created_at, and inbound columns.')

## 9. Resolution Signal Analysis

In [ ]:
if 'df' in dir():
    conv_col = col_map.get('conversation_id')
    text_col = col_map.get('text')
    inbound_col = col_map.get('inbound')
    
    if all(c and c in df.columns for c in [conv_col, text_col]):
        # HEURISTIC analysis - not ground truth
        resolution_signals = []
        
        for conv_id, group in df.groupby(conv_col):
            texts = group[text_col].fillna('').astype(str)
            all_text = ' '.join(texts.str.lower())
            
            signals = {
                'conv_id': conv_id,
                'n_messages': len(group),
                'has_thank_you': any(t in all_text for t in ['thank', 'thanks', 'thx']),
                'has_acknowledgement': any(t in all_text for t in ['resolved', 'fixed', 'solved', 'closed']),
            }
            
            # Check if conversation ends with support response
            if inbound_col and inbound_col in group.columns:
                inbound_mask = group[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
                if len(group) > 0:
                    last_is_support = not inbound_mask.iloc[-1]
                    signals['last_is_support'] = bool(last_is_support)
                else:
                    signals['last_is_support'] = False
            
            resolution_signals.append(signals)
        
        res_df = pd.DataFrame(resolution_signals)
        
        # Aggregate (HEURISTIC - clearly labelled)
        resolution_heuristic = {
            'total_conversations': len(res_df),
            'pct_with_thank_you': round(res_df['has_thank_you'].mean() * 100, 1),
            'pct_with_acknowledgement': round(res_df['has_acknowledgement'].mean() * 100, 1),
            'pct_ending_with_support': round(res_df.get('last_is_support', pd.Series([False])).mean() * 100, 1),
            'note': 'These are HEURISTIC signals, not ground-truth resolution labels',
        }
        eda_stats['resolution_heuristic'] = resolution_heuristic
        
        print('Resolution Signal Analysis (HEURISTIC - not ground truth):')
        print(f'  Total conversations: {resolution_heuristic["total_conversations"]:,}')
        print(f'  Containing thank-you: {resolution_heuristic["pct_with_thank_you"]}%')
        print(f'  Containing resolution language: {resolution_heuristic["pct_with_acknowledgement"]}%')
        print(f'  Ending with support response: {resolution_heuristic["pct_ending_with_support"]}%')
        print(f'\n  NOTE: {resolution_heuristic["note"]}')
    else:
        print('Missing required columns.')

## 10. Text Quality & Noise Analysis

In [ ]:
if 'df' in dir():
    text_col = col_map.get('text')
    inbound_col = col_map.get('inbound')
    
    if text_col and text_col in df.columns:
        # Overall text stats
        text_stats = calculate_text_statistics(df[text_col], label='all_messages')
        eda_stats['text_quality'] = text_stats
        
        print('Text Quality Statistics:')
        for k, v in text_stats.items():
            print(f'  {k}: {v}')
        
        # Noise features
        noise = detect_noise_features(df[text_col])
        eda_stats['noise_features'] = noise
        
        print('\nNoise Features:')
        for k, v in noise.items():
            print(f'  {k}: {v}%')
        
        # Customer vs Support noise
        if inbound_col and inbound_col in df.columns:
            inbound_mask = df[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
            cust_noise = detect_noise_features(df[inbound_mask][text_col])
            supp_noise = detect_noise_features(df[~inbound_mask][text_col])
            
            fig, ax = plt.subplots(figsize=(10, 6))
            compare = pd.DataFrame({'Customer': cust_noise, 'Support': supp_noise}).T
            compare.plot(kind='bar', ax=ax, edgecolor='black')
            ax.set_ylabel('Percentage')
            ax.set_title('Noise Features: Customer vs Support')
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
            plt.tight_layout()
            plt.show()

## 11. Duplicate Analysis

In [ ]:
if 'df' in dir():
    text_col = col_map.get('text')
    id_col = col_map.get('tweet_id')
    
    dup_stats = calculate_duplicate_statistics(
        df,
        text_col=text_col if text_col and text_col in df.columns else None,
        id_col=id_col if id_col and id_col in df.columns else None,
    )
    eda_stats['duplicates'] = dup_stats
    
    print('Duplicate Analysis:')
    for k, v in dup_stats.items():
        print(f'  {k}: {v}')

## 12. Language Analysis

In [ ]:
if 'df' in dir():
    lang_col = col_map.get('language')
    
    if lang_col and lang_col in df.columns:
        lang_dist = df[lang_col].value_counts().head(10)
        eda_stats['language'] = {'distribution': lang_dist.to_dict()}
        
        print('Language Distribution (top 10):')
        display(lang_dist)
        
        fig, ax = plt.subplots(figsize=(10, 5))
        lang_dist.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
        ax.set_ylabel('Count')
        ax.set_title('Language Distribution')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print('Language column not detected in the dataset schema.')
        eda_stats['language'] = {'note': 'Language column not available in dataset schema'}

## 13. URL & External Link Analysis

In [ ]:
if 'df' in dir():
    text_col = col_map.get('text')
    inbound_col = col_map.get('inbound')
    
    if text_col and text_col in df.columns:
        import re
        url_pattern = re.compile(r'https?://\S+|t\.co/\S+')
        
        df['_has_url'] = df[text_col].fillna('').str.contains(url_pattern, regex=True)
        
        overall_url_pct = df['_has_url'].mean() * 100
        print(f'Messages with URLs: {overall_url_pct:.1f}%')
        
        if inbound_col and inbound_col in df.columns:
            inbound_mask = df[inbound_col].astype(str).str.lower().isin(['true', '1', 'yes'])
            cust_url = df[inbound_mask]['_has_url'].mean() * 100
            supp_url = df[~inbound_mask]['_has_url'].mean() * 100
            print(f'  Customer messages with URLs: {cust_url:.1f}%')
            print(f'  Support messages with URLs: {supp_url:.1f}%')
        
        eda_stats['url_analysis'] = {'overall_url_pct': round(overall_url_pct, 2)}
        
        df.drop('_has_url', axis=1, inplace=True, errors='ignore')

## 14. Save EDA Statistics

In [ ]:
if eda_stats:
    output_path = INTERIM_DIR / 'eda_statistics.json'
    with open(output_path, 'w') as f:
        json.dump(eda_stats, f, indent=2, default=str)
    print(f'EDA statistics saved to: {output_path}')
    print(f'\nSections included: {list(eda_stats.keys())}')
else:
    print('No statistics collected. Ensure the dataset is available.')

## 15. Implications for Future Phases

This section documents observations that will influence later engineering decisions.

**Note:** These are observations, not decisions. Phase 4 will use them for brand selection.

In [ ]:
print('=== Implications for Future Phases ===')
print()
print('1. BRAND SELECTION: Use the candidate table above to identify brands with')
print('   sufficient multi-turn conversations and response coverage.')
print()
print('2. INTENT DEFINITION: The noise analysis shows what preprocessing will be')
print('   needed. Intent taxonomy should account for social-media-specific issues.')
print()
print('3. RETRIEVAL: Conversation-length distribution informs how many context')
print('   messages to include in retrieval queries.')
print()
print('4. ESCALATION: The resolution heuristic shows that explicit resolution')
print('   labels are not available. Escalation logic will need to infer this.')
print()
print('5. EVALUATION: The absence of ground-truth resolution labels means we')
print('   cannot directly measure resolution rate. Focus on intent accuracy')
print('   and reply groundedness instead.')
print()
print('6. DATA QUALITY: Document any brands with poor response coverage or')
print('   excessive duplicates as potential issues for later phases.')